# Clustering Modeling

This notebook prepares clustering features from `clustering_master`, evaluates candidate values of `k`, fits KMeans, and saves `clustering_result`.

In [ ]:
import numpy as np
import pandas as pd
import bigframes.pandas as bpd

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler, RobustScaler

PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
ANALYTICS_DATASET_ID = "<YOUR_ANALYTICS_BIGQUERY_DATASET>"

CLUSTERING_MASTER_TABLE = f"{PROJECT_ID}.{ANALYTICS_DATASET_ID}.clustering_master"
CLUSTERING_RESULT_TABLE = f"{PROJECT_ID}.{ANALYTICS_DATASET_ID}.clustering_result"

K_RANGE = range(2, 15)
BEST_K = 8
RANDOM_STATE = 42
N_INIT = 50
SCALING = "standard"
CLIP_QUANTILE = 0.01
LOG_TRANSFORM = True

In [ ]:
def load_clustering_master() -> pd.DataFrame:
    sql = f"SELECT * FROM `{CLUSTERING_MASTER_TABLE}` ORDER BY collection"
    return bpd.read_gbq(sql).to_pandas()

def prepare_clustering_dataset(
    scaling: str = "standard",
    clip_quantile: float = 0.01,
    log_transform: bool = True,
    log_candidates=None,
):
    df = load_clustering_master()
    print(f"Loaded clustering_master: {df.shape}")

    feature_list = [
        "unique_holders", "unique_holder_ratio", "top10_share", "holder_hhi",
        "buyer_hhi", "seller_hhi", "longterm_holder_ratio", "trading_days",
        "active_weeks", "trading_week_span", "trades", "daily_trades",
        "buyer_seller_ratio", "p25_price", "median_price", "p75_price",
        "stddev_price", "diff_price", "description_length", "has_project_url",
        "has_twitter", "has_discord", "has_instagram", "has_telegram",
        "is_erc721", "is_erc1155", "has_erc2981", "royalty_fee_percent",
    ]
    feature_list = [c for c in feature_list if c in df.columns]

    df_raw = df[feature_list].apply(pd.to_numeric, errors="coerce")
    df_raw = df_raw.replace([np.inf, -np.inf], np.nan)
    df_raw = df_raw.fillna(df_raw.median())
    df_raw = df_raw.fillna(0.0)
    df_num = df_raw.copy()

    if log_transform:
        if log_candidates is None:
            log_candidates = [
                "unique_holders", "trading_days", "active_weeks", "trades",
                "daily_trades", "median_price", "p25_price", "p75_price",
                "stddev_price", "diff_price", "holder_hhi", "buyer_hhi", "seller_hhi",
            ]
        for col in log_candidates:
            if col not in df_num.columns:
                continue
            s = df_num[col]
            if s.min() <= -1:
                print(f"Skip log1p for {col} (min={s.min()})")
                continue
            df_num[col] = np.log1p(s)

    if clip_quantile and clip_quantile > 0:
        lo, hi = clip_quantile, 1 - clip_quantile
        for col in df_num.columns:
            q_lo, q_hi = df_num[col].quantile([lo, hi])
            df_num[col] = df_num[col].clip(q_lo, q_hi)

    scaler = StandardScaler() if scaling == "standard" else RobustScaler() if scaling == "robust" else None
    if scaler is not None:
        df_cluster = pd.DataFrame(scaler.fit_transform(df_num), columns=df_num.columns, index=df_num.index)
    else:
        df_cluster = df_num.copy()

    df_meta = df[["collection", "category", "first_trade_regime"]].copy()
    return df_cluster, df_raw, df_meta

In [ ]:
def evaluate_k(df_features, k_range=range(2, 15), random_state=42, n_init=50):
    inertias, sil_scores, db_scores = [], [], []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=n_init)
        labels = kmeans.fit_predict(df_features)
        inertias.append(kmeans.inertia_)
        sil_scores.append(silhouette_score(df_features, labels))
        db_scores.append(davies_bouldin_score(df_features, labels))
        print(f"k={k:2d} | inertia={inertias[-1]:.2f} | silhouette={sil_scores[-1]:.4f} | DBI={db_scores[-1]:.4f}")
    return pd.DataFrame({"k": list(k_range), "inertia": inertias, "silhouette": sil_scores, "dbi": db_scores})

def cluster_centroid_summary(df_features, labels):
    df = df_features.copy()
    df["cluster"] = labels
    centroid_table = df.groupby("cluster").mean().sort_index(axis=1)
    print("Cluster centroid summary:")
    print(centroid_table)
    return centroid_table

def compute_distance_to_centroid(df_features, labels, collection_ids=None):
    centroid_table = cluster_centroid_summary(df_features, labels)
    X = df_features.values
    centers = centroid_table.values
    distances = np.linalg.norm(X - centers[labels], axis=1)
    if collection_ids is None:
        collection_ids = df_features.index.astype(str)
    df_dist = pd.DataFrame({
        "collection": collection_ids,
        "cluster": labels.astype(int),
        "distance_to_centroid": distances,
    })
    return df_dist, centroid_table

In [ ]:
X_cluster, X_raw, X_meta = prepare_clustering_dataset(
    scaling=SCALING,
    clip_quantile=CLIP_QUANTILE,
    log_transform=LOG_TRANSFORM,
)

k_eval = evaluate_k(X_cluster, k_range=K_RANGE, random_state=RANDOM_STATE, n_init=N_INIT)
print(k_eval)

kmeans = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=N_INIT)
labels = kmeans.fit_predict(X_cluster)

df_dist, centroid_table = compute_distance_to_centroid(
    df_features=X_cluster,
    labels=labels,
    collection_ids=X_meta["collection"],
)

print(df_dist.head())
print(centroid_table.head())

In [ ]:
bf_dist = bpd.DataFrame(df_dist)
bf_dist.to_gbq(CLUSTERING_RESULT_TABLE, if_exists="replace")
print(f"Saved clustering result -> {CLUSTERING_RESULT_TABLE}")